# Ticketing AI Assistant — the Python workflow, step by step

This notebook walks the agent workflow from configuration to a finished answer,
using **your real model and credentials** — nothing here is mocked.

Each step is the smallest thing that can actually run on its own, so if a step
breaks you know exactly where. The running order matches how `api.py` wires it
together in production; this is the same code path, minus the HTTP layer.

### What you need running first

`OpsLookup.Api` on port **5250** — it owns SQL Server. This notebook cannot start it.

```powershell
cd src\OpsLookup.Api
dotnet run
```

### How to run

Select the **`opslookup (Python 3.14)`** kernel. If it is missing:

```powershell
cd src\OpsLookup.Graph
.\.venv\Scripts\pip install -r requirements-dev.txt
.\.venv\Scripts\python -m ipykernel install --user --name opslookup --display-name "opslookup (Python 3.14)"
```

Run all cells from the top. Order matters — later steps build on objects earlier
steps created.

### Two rules this notebook follows

- **No `asyncio.run()` in a cell.** Jupyter already runs an event loop, so calling
  it raises `RuntimeError: cannot be called from a running event loop`. Turns below
  use top-level `await` instead.
- **No secrets in output.** The service token and model key are shown as
  lengths, never values.


## Step 0 — Configuration and credentials

Two credentials are needed:

| Credential | Source | Used for |
|---|---|---|
| `LONGCAT_API_KEY` | `.env` | Calling the model |
| API bearer token | **minted live below** | Calling `OpsLookup.Api` |

The service token is minted at runtime on purpose. Dev tokens expire after
**60 minutes**, so a token pasted into `.env` by hand is a token that will be
dead the next morning. Minting it here means this notebook always works, at the
cost of one extra request.

`config.py` calls `load_dotenv()` itself, so importing it is enough to pick up
`.env` — but only if the kernel's working directory is this project. That is what
the `sys.path` line below guarantees.


In [1]:
import json
import sys
from dataclasses import replace
from pathlib import Path

import httpx

# The kernel's CWD is wherever Jupyter was started, not necessarily the project.
# `.env` and the `config`/`tools` packages are resolved relative to this file, so
# pin both to the project directory before importing anything local.
PROJECT = Path.cwd()
if not (PROJECT / "config.py").exists():
    PROJECT = Path(r"D:\Projects\Operation RAG\src\OpsLookup.Graph")
    sys.path.insert(0, str(PROJECT))

import os
os.chdir(PROJECT)          # so load_dotenv() finds .env
sys.path.insert(0, str(PROJECT))

print("project:", PROJECT)
print("config.py found:", (PROJECT / "config.py").exists())
print(".env found:", (PROJECT / ".env").exists())


project: D:\Projects\Operation RAG\src\OpsLookup.Graph
config.py found: True
.env found: True


In [2]:
# Mint a fresh service token. /auth/dev-token is Development-only and unauthenticated
# by design, which is why it must never exist in a Production environment.
base = os.getenv("API_BASE_URL", "http://localhost:5250").rstrip("/")

with httpx.Client(timeout=30) as client:
    api_health = client.get(f"{base}/health").json()
    resp = client.post(f"{base}/auth/dev-token", params={"operatorId": "notebook"})
    resp.raise_for_status()
    minted_token = resp.json()["accessToken"]

print("api health :", api_health)
print("token      :", minted_token[:12] + "...", f"({len(minted_token)} chars)")


api health : {'status': 'ok'}
token      : eyJhbGciOiJI... (253 chars)


In [3]:
from config import load_settings

# load_settings() requires API_TOKEN. Replace it with the token just minted, so a
# stale value in .env cannot make this notebook fail with an opaque 401 later.
raw = load_settings()
settings = replace(raw, api_token=minted_token)

print("model          :", settings.longcat_model)
print("model base url :", settings.longcat_base_url)
print("api base url   :", settings.api_base_url)
print("operator id    :", settings.operator_id)
print("recursion limit:", settings.recursion_limit)
print()
print("gateway headers (values redacted):")
for key, value in settings.default_headers().items():
    print(f"  {key}: {value[:14]}... ({len(value)} chars)" if len(value) > 20 else f"  {key}: {value}")


model          : longcat-2.5-preview-free
model base url : https://opencode.ai/zen/go/v1
api base url   : http://localhost:5250
operator id    : op-4471
recursion limit: 10

gateway headers (values redacted):
  User-Agent: OpsLookup.Grap... (45 chars)
  x-opencode-session: opslookup-a14d... (26 chars)


The two headers in that last block are why this works at all. The OpenCode Zen
gateway rejects a request that does not carry `x-opencode-session` and a
`User-Agent`, returning `MissingSessionID`. A plain OpenAI client sends neither,
so `llm.py` attaches them explicitly. If you ever see `MissingSessionID`, this is
the thing to check.


## Step 1 — The API client

`LookupApiClient` is the only thing in Python that talks to `.NET`. It holds an
HTTP client and a bearer token, and **no database driver** — Python cannot reach
SQL Server even by accident, which is the point of the split.


In [4]:
from tools.api_client import LookupApiClient

client = LookupApiClient(
    base_url=settings.api_base_url,
    token=settings.api_token,
    operator_id=settings.operator_id,
    timeout_seconds=settings.request_timeout_seconds,
)

print("base url:", settings.api_base_url)
print("operator:", settings.operator_id)
print("reachable:", await client.health())


base url: http://localhost:5250
operator: op-4471
reachable: True


## Step 2 — The tools

Two typed tools. The model chooses among them and supplies typed arguments; it
never emits SQL, a table name, or a filter expression.

| Tool | Searches by |
|---|---|
| `get_contract` | social ID (required), party name (optional) |
| `get_attorney` | social ID **or** party name, or both |


In [5]:
from tools.attorneys import build_attorney_tools
from tools.contracts import build_tools

tools = build_tools(client) + build_attorney_tools(client)

for tool in tools:
    print(f"{tool.name}")
    print(f"    {tool.description[:100]}...")
    props = tool.args_schema.model_json_schema()["properties"]
    for prop, spec in props.items():
        print(f"      - {prop}: {spec.get('type', spec.get('anyOf'))}")
    print()


get_contract
    Look up contract records for a party by their social ID (must be 10 digits, starting with 1 or 2). U...
      - social_id: integer
      - name: [{'type': 'string'}, {'type': 'null'}]

get_attorney
    Look up the attorney acting for a party, and whether that attorney's status is approved or faked. Se...
      - social_id: [{'type': 'integer'}, {'type': 'null'}]
      - party_name: [{'type': 'string'}, {'type': 'null'}]



Note the shape difference. `get_contract` has a **required** `social_id`; the
attorney tool has two optional arguments and refuses to run with neither, because
with no filter at all it would return every attorney on file — a data dump rather
than a lookup.


## Step 3 — Call the data directly, with no model involved

Do this before involving the LLM. If a lookup is wrong here, no amount of prompt
tuning will fix it, and you want to know that before you start blaming the model.

This is also the grounding check that the whole app depends on: these rows are the
only facts the agent is ever allowed to assert.


In [6]:
contracts = await client.get_contracts(social_id=1234567890)
attorneys = await client.get_attorneys(social_id=1987654321)

print(f"contracts for 1234567890 ({len(contracts)}):")
for row in contracts:
    print("   ", row)

print(f"\nattorney for 1987654321 ({len(attorneys)}):")
for row in attorneys:
    print("   ", row)


contracts for 1234567890 (2):
    {'contractCode': 100234, 'socialId': 1234567890, 'partyName': 'A Nasser', 'contractName': 'Annual Maintenance Agreement'}
    {'contractCode': 100235, 'socialId': 1234567890, 'partyName': 'A Nasser', 'contractName': 'Network Support Retainer'}

attorney for 1987654321 (1):
    {'attorneyId': 1, 'attorneyName': 'Lena Fischer', 'status': 'approved', 'partyId': 3, 'partyName': 'B Karimi', 'socialId': 1987654321}


In [7]:
# The empty result is the case worth staring at. It means "no attorney on file
# for this party", which is a completely different answer from "the attorney is
# approved". The UI distinguishes them; the model is told to as well.
none_on_file = await client.get_attorneys(social_id=1234567890)
print("party with no attorney ->", none_on_file)
print("interpretation: no attorney on file, NOT 'not approved'")


party with no attorney -> []
interpretation: no attorney on file, NOT 'not approved'


## Step 4 — The model

`build_llm` binds the tools to a `ChatOpenAI` client pointed at LongCat via the
Zen gateway. Binding the tools is what makes tool calling possible at all — an
unbound model cannot emit a tool call, and every answer becomes a guess.


In [8]:
from llm import build_llm

llm = build_llm(settings, tools)
print("model:", settings.longcat_model)
print("thinking:", "disabled (fast; this is extraction plus a short answer)")
print("temperature: 0")
print("timeout: 90s, max_retries: 2")
print("tools bound:", len(tools))


model: longcat-2.5-preview-free
thinking: disabled (fast; this is extraction plus a short answer)
temperature: 0
timeout: 90s, max_retries: 2
tools bound: 2


## Step 5 — The graph

The loop is:

```
START → llm → (tool calls? → action → back to llm) → plain text → END
```

Each pass through `llm` is one round trip to the model. So a ticket needing **N**
API calls costs **N + 1** model calls and `2N + 1` graph steps — all inside **one
user turn**.

That arithmetic matters, because `RECURSION_LIMIT` (10) is the ceiling: N=4 needs
9 steps and fits; N=5 needs 11 and fails with `GraphRecursionError`.


In [9]:
from langgraph.checkpoint.memory import InMemorySaver

from graph.build import any_tool_error, build_graph, system_prompt

# InMemorySaver here rather than the SQLite checkpointer: it keeps this notebook
# from writing a checkpoints.sqlite next to your project. The production wiring in
# api.py uses AsyncSqliteSaver, which behaves the same for a single process.
graph = build_graph(llm, tools, InMemorySaver())

print("nodes: llm, action")
print("edges: START→llm, llm→(action|END), action→llm")
print()
print("steps needed for N API calls = 2N + 1, ceiling at recursion_limit =",
      settings.recursion_limit)
for n in range(1, 6):
    print(f"   N={n}: {2*n+1:2d} steps -> "
          + ("fits" if 2*n+1 <= settings.recursion_limit else "GraphRecursionError"))


nodes: llm, action
edges: START→llm, llm→(action|END), action→llm

steps needed for N API calls = 2N + 1, ceiling at recursion_limit = 10
   N=1:  3 steps -> fits
   N=2:  5 steps -> fits
   N=3:  7 steps -> fits
   N=4:  9 steps -> fits
   N=5: 11 steps -> GraphRecursionError


## Step 6 — A real turn

Everything above was setup. This is the actual agent answering a question, with
your model and real data.


In [12]:
async def run_turn(graph, question, *, locale="en", thread_id=None, limit=None):
    """One turn, returning the graph result rather than just the text.

    Mirrors host.py::_run_turn so what you see here is what the .NET host runs.
    """
    thread_id = thread_id or f"nb-{abs(hash(question)) % 10_000}"
    result = await graph.ainvoke(
        {
            "messages": [("user", question)],
            "ticket_id": thread_id,
            "operator_id": settings.operator_id,
            "locale": locale,
        },
        {
            "configurable": {"thread_id": thread_id},
            "recursion_limit": limit or settings.recursion_limit,
        },
    )
    return result, thread_id


result, thread_id = await run_turn(
    graph, "What contracts and attornys are held for social ID 1234567890?"
)
print("answer:\n", result["messages"][-1].content)


answer:
 Here's what I found for social ID 1234567890 (party name: A Nasser):

**Contracts (2):**
1. Annual Maintenance Agreement (contract code 100234)
2. Network Support Retainer (contract code 100235)

**Attorney:**
No attorney is on file for this party.


## Step 7 — Reading the trace

The answer alone does not tell you whether it was *verified*. The message list
does. Walk it and you can see exactly what the model asked for and what came back.


In [13]:
def show_trace(result, *, max_len=110):
    """Print the message sequence the graph produced."""
    tool_calls = sum(len(getattr(m, "tool_calls", []) or []) for m in result["messages"])
    print(f"{'#':<3}{'type':<14}{'detail'}")
    print("-" * 92)
    for i, m in enumerate(result["messages"]):
        kind = type(m).__name__
        if kind == "HumanMessage":
            detail = str(m.content)[:max_len]
        elif kind == "ToolMessage":
            detail = f"{m.name or ''} -> {str(m.content)[:max_len]}"
        elif getattr(m, "tool_calls", None):
            calls = ", ".join(
                f"{c['name']}({json.dumps(c['args'])})" for c in m.tool_calls
            )
            detail = f"requested {calls}"
        else:
            detail = str(m.content)[:max_len].replace("\n", " ")
        print(f"{i:<3}{kind:<14}{detail}")
    print("-" * 92)
    print(f"tool_calls = {tool_calls}")
    return tool_calls


tool_calls = show_trace(result)


#  type          detail
--------------------------------------------------------------------------------------------
0  HumanMessage  What contracts and attornys are held for social ID 1234567890?
1  AIMessage     requested get_contract({"social_id": 1234567890}), get_attorney({"social_id": 1234567890})
2  ToolMessage   get_contract -> {"social_id": 1234567890, "results": [{"contractCode": 100234, "socialId": 1234567890, "partyName": "A Nasser"
3  ToolMessage   get_attorney -> {"results": []}
4  AIMessage     Here's what I found for social ID 1234567890 (party name: A Nasser):  **Contracts (2):** 1. Annual Maintenance
--------------------------------------------------------------------------------------------
tool_calls = 2


A healthy trace alternates: the model asks, the tool answers, the model asks again
or concludes. If you see `HumanMessage` immediately followed by the final
`AIMessage` with no `ToolMessage` in between, the model answered from its own
knowledge — `tool_calls = 0`, and the answer is a guess.


## Step 8 — The turn contract

What the .NET host returns to the browser, and the two fields that decide whether
an answer may be trusted:

| Field | Meaning |
|---|---|
| `toolCalls` | how many tools were **invoked** |
| `lookupFailed` | whether one of them **failed** |

`toolCalls` alone is not enough. A lookup that the API rejected still counts 1, so
a UI that badges on `toolCalls` alone will show a green "read from the database"
badge for an answer where nothing was read. `any_tool_error` closes that gap.


In [14]:
print("toolCalls      :", tool_calls)
print("lookupFailed   :", any_tool_error(result["messages"]))
print("grounded       :", tool_calls > 0 and not any_tool_error(result["messages"]))


toolCalls      : 2
lookupFailed   : False
grounded       : True


In [15]:
# The counter-intuitive case, demonstrated rather than asserted: a tool error is
# still a tool call. Both tools return failures as {"error": ...} text so the
# model can explain them, which means "was it read from the database" has to be
# answered by looking at the content, not by counting.
ok_tool  = type(result["messages"][2])(content='{"results": [{"contractCode": 100234}]}', tool_call_id="a")
bad_tool = type(result["messages"][2])(content='{"error": "Social ID must be 10 digits starting with 1 or 2."}', tool_call_id="b")

from langchain_core.messages import HumanMessage
print("successful tool -> lookupFailed:", any_tool_error([HumanMessage("q"), ok_tool]))
print("failed tool     -> lookupFailed:", any_tool_error([HumanMessage("q"), bad_tool]))


successful tool -> lookupFailed: False
failed tool     -> lookupFailed: True


## Step 9 — Arabic

The page sends its language, and the locale selects the system prompt, so the
answer comes back in the language the member is reading. Nothing else changes:
same tools, same data, same graph.

Two things stay in Latin digits on purpose — social IDs and contract codes — and
database values (`faked`, `Annual Maintenance Agreement`) are never translated.


In [16]:
for loc in ("ar", "en"):
    print(f"--- system_prompt({loc!r}) ---")
    print(system_prompt(loc).splitlines()[0])
    print()

ar_result, _ = await run_turn(
    graph, "هل المحامي للرقم الاجتماعي 2111111111 معتمد؟", locale="ar", thread_id="nb-ar"
)
print("answer (ar):\n", ar_result["messages"][-1].content)


--- system_prompt('ar') ---
أنت مساعد دعم لفريق تشغيل تقنية المعلومات.

--- system_prompt('en') ---
You are a support assistant for IT operations staff.

answer (ar):
 المحامي المسجّل لهذا العميل هو **Marcus Webb**، وحالته **faked** — أي أنه مُطعن فيه بالاحتيال، وليس معتمدًا.


## Step 10 — A follow-up keeps its context

The checkpointer replays the thread, which is what lets the agent ask a clarifying
question and then continue. Note that `toolCalls` counts *every* tool call on the
thread, not just this turn — so a follow-up that needs no new lookup still reports
the earlier ones. That is intentional for the transcript, and the reason
`any_tool_error` is scoped to the current turn rather than the whole history.


In [18]:
follow_up, _ = await run_turn(
    graph, "And who is their attorney?", thread_id=thread_id
)
print("follow-up answer:\n", follow_up["messages"][-1].content)
print()
human = [m.content for m in follow_up["messages"] if type(m).__name__ == "HumanMessage"]
print("user turns on this thread:", len(human))
for turn in human:
    print("   -", turn)


follow-up answer:
 I checked again — there is no attorney on file for this party (social ID 1234567890, A Nasser). The system returned no attorney record for them.

user turns on this thread: 3
   - What contracts and attornys are held for social ID 1234567890?
   - And who is their attorney?
   - And who is their attorney?


## Step 11 — The three failure modes worth knowing

| Input | Correct behaviour |
|---|---|
| valid social ID, no rows | say nothing was found — do not speculate |
| malformed social ID | the API returns 400; the model explains, rather than inventing a result |
| missing social ID | ask for it — never guess |

The third is the one that matters most, and it is enforced by the system prompt
rather than by code, so it is worth seeing that it actually holds.


In [19]:
cases = [
    ("valid ID, no such customer", "What contracts are held for social ID 1999999999?"),
    ("malformed social ID",       "What contracts are held for social ID 345?"),
    ("no social ID at all",       "Can you check what contracts this customer holds?"),
]

for label, question in cases:
    res, _ = await run_turn(graph, question, thread_id=f"nb-fail-{abs(hash(label))%9999}")
    answer = res["messages"][-1].content
    failed = any_tool_error(res["messages"])
    print(f"--- {label} ---")
    print(answer.strip()[:300])
    print(f"[lookupFailed={failed}]\n")


--- valid ID, no such customer ---
No contracts are on file for social ID 1999999999. The search came back empty, so this party doesn't appear to hold any contracts in our records.
[lookupFailed=False]

--- malformed social ID ---
I can't look that up with "345" — a social ID needs to be exactly 10 digits and start with a 1 or a 2.

Could you check the number with the member and send over the full 10-digit social ID? Once I have that I'll pull the contracts right away.
[lookupFailed=False]

--- no social ID at all ---
I can help with that. To look up the contracts, I'll need the customer's social ID — it's a 10-digit number starting with 1 or 2.

Could you share that?
[lookupFailed=False]



## Step 12 — Where the ceiling is

Worth restating at the end, because it is the one limit that turns a working
ticket into an unexplained failure. A ticket needing more API calls than the
recursion limit allows does not degrade gracefully — it returns
`GraphRecursionError`, and everything it had already retrieved is discarded.

Raise `RECURSION_LIMIT` in `.env` if tickets in your environment routinely need
five or more lookups.


In [20]:
limit = settings.recursion_limit
headroom = (limit - 1) // 2
print(f"RECURSION_LIMIT = {limit}")
print(f"max API calls per turn = {headroom}")
print(f"one more lookup beyond that fails the whole turn.")


RECURSION_LIMIT = 10
max API calls per turn = 4
one more lookup beyond that fails the whole turn.


## Where to go next

| To check | Look at |
|---|---|
| The .NET API end to end | `Check API.ipynb` |
| Wiring without a model key | `smoke_test.py` — stubs the LLM entirely |
| This workflow, in the embedded host | `src/OpsLookup.NetCorePythonNetApp/Python/host.py` |

The gap between this and a full evaluation suite is a golden dataset: a set of
questions with their mechanically checkable expected answers. Everything the
workflow does can be asserted; only the phrasing of the reply needs a human.
